# EDA Recommender Splits

Split analysis for recommender datasets.

Steps:
- Load a MovieLens sample or catalog fallback.
- Split ratings into train/val/test.
- Summarize user/item coverage per split.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

rec_root = REPO_ROOT / 'data' / 'raw' / 'recommendation'
movielens_path = rec_root / 'movielens.csv'
items_path = rec_root / 'items.csv'

summary = {
    'source': {},
    'splits': {},
}

if movielens_path.exists():
    size_mb = movielens_path.stat().st_size / 1024**2
    max_rows = 1_000_000 if size_mb > 200 else None
    df = pd.read_csv(movielens_path, nrows=max_rows)
    summary['source']['path'] = str(movielens_path)
    summary['source']['rows'] = int(df.shape[0])
    summary['source']['sampled'] = max_rows is not None
    print('MovieLens rows:', df.shape[0], 'sampled:', max_rows is not None)
else:
    print('Missing movielens.csv, using items catalog')
    df = pd.read_csv(items_path)
    summary['source']['path'] = str(items_path)
    summary['source']['rows'] = int(df.shape[0])

if 'rating' in df.columns:
    train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    summary['splits'] = {
        'train': int(train_df.shape[0]),
        'val': int(val_df.shape[0]),
        'test': int(test_df.shape[0]),
    }
    print('Split sizes:', summary['splits'])

    for name, split_df in [('train', train_df), ('val', val_df), ('test', test_df)]:
        split_summary = {}
        for col in ['userId', 'movieId']:
            if col in split_df.columns:
                split_summary[f'{col}_unique'] = int(split_df[col].nunique())
        if 'rating' in split_df.columns:
            split_summary['rating_counts'] = split_df['rating'].value_counts().head(10).to_dict()
        summary['splits_detail'] = summary.get('splits_detail', {})
        summary['splits_detail'][name] = split_summary
        print(name, split_summary)
else:
    print('No rating column to split; catalog only')


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_recommender_splits_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize recommender-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'recommender' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No recommender entries found in TRAINING_DATA.json')
    else:
        print('recommender datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
